<a href="https://colab.research.google.com/github/Caleb-John1603/AI_5th_sem/blob/main/1BM24CS095_WEEK%209_Resolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#!/usr/bin/env python3
"""
First-Order Logic resolution prover
===================================
  1. parses premises + goal written in FOL
  2. converts everything to CNF (eliminate <->/->, NNF, standardise variables,
     Skolemise, drop quantifiers, distribute | over &)
  3. negates the goal and runs resolution refutation (set-of-support, unit
     preference, factoring, full unification with occurs check)
  4. prints the proof and draws it as a graph that starts at the NEGATED GOAL
     and ends at the EMPTY CLAUSE  (PNG via matplotlib + Graphviz .dot file)

SYNTAX
  predicates / constants / functions : Capitalised    Man(x)  Loves(x, Mother(y))  Marcus
  variables                          : lowercase      x  y  z1
  not  ~      and  &      or  |      implies  ->      iff  <->
  quantifiers: forall x (...)   exists y (...)   forall x,y (...)   forall x forall y ...
  (a quantifier's scope extends as far right as possible, so write
   (exists y P(y)) -> Q   or   forall x ((forall y R(x,y)) -> S(x))   to limit it;
   the "parsed as:" line printed for every formula shows how it was read;
   unicode symbols  ∀ ∃ ¬ ∧ ∨ → ↔  are accepted too)
  Free variables: universally quantified in premises, existentially in the goal.

USAGE
  python fol_resolution.py                 # interactive
  python fol_resolution.py --demo          # built-in example
  python fol_resolution.py --file kb.txt   # one premise per line, last line  GOAL: <formula>
  options: --out proof.png  --show  --max-clauses 3000
"""
import argparse
import heapq
import itertools
import re
import sys
import textwrap
from collections import defaultdict


# ════════════════════════════════════════════════════════════════════
#  1.  PARSER      (AST: tuples)
#      terms   : ("v", name)  |  ("f", name, (args...))      (constant = f with no args)
#      formula : ("pred", name, args) | ("not", f) | ("and"/"or"/"imp"/"iff", f, g)
#                | ("forall"/"exists", var, f)
# ════════════════════════════════════════════════════════════════════
class ParseError(Exception):
    pass


SYMBOLS = [("<=>", "<->"), ("=>", "->"), ("∀", " forall "), ("∃", " exists "),
           ("¬", "~"), ("∧", "&"), ("∨", "|"), ("↔", "<->"), ("⇔", "<->"),
           ("→", "->"), ("⇒", "->")]
TOKEN_RE = re.compile(r"\s*(<->|->|[A-Za-z0-9_']+|[()~&|,.])")
QUANTS = ("forall", "exists")


def tokenize(text):
    for a, b in SYMBOLS:
        text = text.replace(a, b)
    text, pos, toks = text.strip(), 0, []
    while pos < len(text):
        m = TOKEN_RE.match(text, pos)
        if not m:
            raise ParseError(f"Unexpected character {text[pos]!r} in: {text}")
        toks.append(m.group(1))
        pos = m.end()
    return toks


def is_var_name(s):
    return s[0].islower() and s not in QUANTS


class Parser:
    def __init__(self, text):
        self.text, self.toks, self.i = text, tokenize(text), 0

    def peek(self):
        return self.toks[self.i] if self.i < len(self.toks) else None

    def next(self):
        t = self.peek()
        if t is None:
            raise ParseError(f"Unexpected end of formula: {self.text}")
        self.i += 1
        return t

    def expect(self, t):
        got = self.next()
        if got != t:
            raise ParseError(f"Expected {t!r} but found {got!r} in: {self.text}")

    def parse(self):
        f = self.iff()
        if self.peek() is not None:
            raise ParseError(f"Unexpected {self.peek()!r} in: {self.text}")
        return f

    def iff(self):
        l = self.imp()
        while self.peek() == "<->":
            self.next()
            l = ("iff", l, self.imp())
        return l

    def imp(self):
        l = self.disj()
        if self.peek() == "->":
            self.next()
            return ("imp", l, self.imp())          # right-associative
        return l

    def disj(self):
        l = self.conj()
        while self.peek() == "|":
            self.next()
            l = ("or", l, self.conj())
        return l

    def conj(self):
        l = self.unary()
        while self.peek() == "&":
            self.next()
            l = ("and", l, self.unary())
        return l

    def unary(self):
        t = self.peek()
        if t == "~":
            self.next()
            return ("not", self.unary())
        if t in QUANTS:
            self.next()
            vs = []
            while True:
                v = self.next()
                if not is_var_name(v):
                    raise ParseError(f"Bad variable name {v!r} after quantifier in: {self.text}")
                vs.append(v)
                if self.peek() == ",":
                    self.next()
                    continue
                nxt = self.peek()
                if nxt is not None and is_var_name(nxt):
                    continue
                break
            if self.peek() == ".":
                self.next()
            body = self.iff()                       # widest scope
            for v in reversed(vs):
                body = (t, v, body)
            return body
        if t == "(":
            self.next()
            f = self.iff()
            self.expect(")")
            return f
        return self.atom()

    def atom(self):
        name = self.next()
        if not name[0].isupper():
            raise ParseError(f"Predicate names must start with a capital letter, got {name!r} in: {self.text}")
        args = []
        if self.peek() == "(":
            self.next()
            if self.peek() != ")":
                args.append(self.term())
                while self.peek() == ",":
                    self.next()
                    args.append(self.term())
            self.expect(")")
        return ("pred", name, tuple(args))

    def term(self):
        name = self.next()
        if not re.fullmatch(r"[A-Za-z0-9_']+", name):
            raise ParseError(f"Bad term {name!r} in: {self.text}")
        if self.peek() == "(":
            self.next()
            args = [self.term()]
            while self.peek() == ",":
                self.next()
                args.append(self.term())
            self.expect(")")
            return ("f", name, tuple(args))
        if name[0].islower():
            return ("v", name)
        return ("f", name, ())


# ════════════════════════════════════════════════════════════════════
#  2.  CNF CONVERSION
# ════════════════════════════════════════════════════════════════════
def term_vars(t):
    if t[0] == "v":
        return {t[1]}
    return set().union(*[term_vars(a) for a in t[2]]) if t[2] else set()


def free_vars(f, bound=frozenset()):
    k = f[0]
    if k == "pred":
        return {v for a in f[2] for v in term_vars(a) if v not in bound}
    if k == "not":
        return free_vars(f[1], bound)
    if k in ("and", "or", "imp", "iff"):
        return free_vars(f[1], bound) | free_vars(f[2], bound)
    return free_vars(f[2], bound | {f[1]})


def nnf(f, neg=False):
    """Eliminates -> and <->, pushes negation down to the atoms."""
    k = f[0]
    if k == "pred":
        return ("not", f) if neg else f
    if k == "not":
        return nnf(f[1], not neg)
    if k == "and":
        return ("or" if neg else "and", nnf(f[1], neg), nnf(f[2], neg))
    if k == "or":
        return ("and" if neg else "or", nnf(f[1], neg), nnf(f[2], neg))
    if k == "imp":
        return nnf(("or", ("not", f[1]), f[2]), neg)
    if k == "iff":
        return nnf(("and", ("imp", f[1], f[2]), ("imp", f[2], f[1])), neg)
    if k == "forall":
        return ("exists" if neg else "forall", f[1], nnf(f[2], neg))
    if k == "exists":
        return ("forall" if neg else "exists", f[1], nnf(f[2], neg))
    raise ValueError(k)


def subst_term(t, env):
    if t[0] == "v":
        return env.get(t[1], t)
    return ("f", t[1], tuple(subst_term(a, env) for a in t[2]))


class Converter:
    """Standardises variables apart + Skolemises (counters shared across all formulas)."""

    def __init__(self):
        self.n_var = 0
        self.n_sk = 0

    def skolemize(self, f, env, univ):
        k = f[0]
        if k == "pred":
            return ("pred", f[1], tuple(subst_term(a, env) for a in f[2]))
        if k == "not":
            return ("not", self.skolemize(f[1], env, univ))
        if k in ("and", "or"):
            return (k, self.skolemize(f[1], env, univ), self.skolemize(f[2], env, univ))
        if k == "forall":
            self.n_var += 1
            nv = ("v", f"u{self.n_var}")
            return self.skolemize(f[2], {**env, f[1]: nv}, univ + [nv])
        if k == "exists":
            self.n_sk += 1
            sk = ("f", f"Sk{self.n_sk}", tuple(univ))     # Skolem constant / function
            return self.skolemize(f[2], {**env, f[1]: sk}, univ)
        raise ValueError(k)


def literal(f):
    """literal = (sign, predicate-name, args) ; sign True = positive"""
    if f[0] == "pred":
        return (True, f[1], f[2])
    return (False, f[1][1], f[1][2])


def to_clauses(f):
    """Distribute OR over AND on a quantifier-free NNF formula -> list of frozensets."""
    if f[0] == "and":
        return to_clauses(f[1]) + to_clauses(f[2])
    if f[0] == "or":
        return [a | b for a in to_clauses(f[1]) for b in to_clauses(f[2])]
    return [frozenset([literal(f)])]


def close(f, quant):
    for v in sorted(free_vars(f), reverse=True):
        f = (quant, v, f)
    return f


def premise_to_clauses(f, conv):
    return to_clauses(conv.skolemize(nnf(close(f, "forall")), {}, []))


def negated_goal_to_clauses(f, conv):
    return to_clauses(conv.skolemize(nnf(("not", close(f, "exists"))), {}, []))


# ════════════════════════════════════════════════════════════════════
#  3.  TERMS, UNIFICATION, CLAUSES
# ════════════════════════════════════════════════════════════════════
def walk(t, s):
    while t[0] == "v" and t[1] in s:
        t = s[t[1]]
    return t


def apply(t, s):
    if t[0] == "v":
        return apply(s[t[1]], s) if t[1] in s else t
    return ("f", t[1], tuple(apply(a, s) for a in t[2]))


def bind(v, t, s):
    if v[1] in term_vars(apply(t, s)):          # occurs check
        return None
    s2 = dict(s)
    s2[v[1]] = t
    return s2


def unify(a, b, s):
    a, b = walk(a, s), walk(b, s)
    if a == b:
        return s
    if a[0] == "v":
        return bind(a, b, s)
    if b[0] == "v":
        return bind(b, a, s)
    if a[1] == b[1] and len(a[2]) == len(b[2]):
        for x, y in zip(a[2], b[2]):
            s = unify(x, y, s)
            if s is None:
                return None
        return s
    return None


def unify_args(args1, args2):
    s = {}
    for x, y in zip(args1, args2):
        s = unify(x, y, s)
        if s is None:
            return None
    return s


def canonical(lits, prefix="x"):
    """Rename variables to prefix1, prefix2, ... (by order of appearance)."""
    ordered = sorted(lits, key=lambda l: (l[1], l[0], repr(l[2])))
    mapping = {}

    def rn(t):
        if t[0] == "v":
            if t[1] not in mapping:
                mapping[t[1]] = ("v", f"{prefix}{len(mapping) + 1}")
            return mapping[t[1]]
        return ("f", t[1], tuple(rn(a) for a in t[2]))

    return frozenset((sg, n, tuple(rn(a) for a in args)) for sg, n, args in ordered)


def is_tautology(lits):
    return any((not sg, n, args) in lits for sg, n, args in lits)


def subst_lits(lits, s):
    return frozenset((sg, n, tuple(apply(a, s) for a in args)) for sg, n, args in lits)


# ── pretty printing ─────────────────────────────────────────────────
def fmt_term(t):
    if t[0] == "v" or not t[2]:
        return t[1]
    return f"{t[1]}({','.join(fmt_term(a) for a in t[2])})"


def fmt_lit(l, uni=False):
    sg, n, args = l
    s = n + (f"({','.join(fmt_term(a) for a in args)})" if args else "")
    return s if sg else ("¬" if uni else "~") + s


def fmt_clause(lits, uni=False):
    if not lits:
        return "□" if uni else "[] (empty clause)"
    ordered = sorted(lits, key=lambda l: (l[1], not l[0], repr(l[2])))
    return (" ∨ " if uni else " | ").join(fmt_lit(l, uni) for l in ordered)


def fmt_formula(f):
    """Fully parenthesised rendering - shows exactly how the input was parsed."""
    k = f[0]
    if k == "pred":
        return f[1] + (f"({','.join(fmt_term(a) for a in f[2])})" if f[2] else "")
    if k == "not":
        return "~" + fmt_formula(f[1])
    if k in ("and", "or", "imp", "iff"):
        op = {"and": "&", "or": "|", "imp": "->", "iff": "<->"}[k]
        return f"({fmt_formula(f[1])} {op} {fmt_formula(f[2])})"
    return f"({k} {f[1]} {fmt_formula(f[2])})"


def fmt_theta(s):
    return "{" + ", ".join(f"{v}/{fmt_term(apply(t, s))}" for v, t in s.items()) + "}"


# ════════════════════════════════════════════════════════════════════
#  4.  RESOLUTION
# ════════════════════════════════════════════════════════════════════
class Clause:
    def __init__(self, cid, lits, parents, kind, note, theta, sos):
        self.id, self.lits, self.parents = cid, lits, parents
        self.kind, self.note, self.theta, self.sos = kind, note, theta, sos


def resolvents(c1, c2):
    """All binary resolvents of clause c1 with clause c2 (c2's variables renamed apart)."""
    c2 = canonical(c2, "y")
    for l1 in c1:
        for l2 in c2:
            if l1[0] != l2[0] and l1[1] == l2[1] and len(l1[2]) == len(l2[2]):
                s = unify_args(l1[2], l2[2])
                if s is None:
                    continue
                rest = subst_lits((c1 - {l1}) | (c2 - {l2}), s)
                yield l1[1], s, rest


def factors(lits):
    L = list(lits)
    for i in range(len(L)):
        for j in range(i + 1, len(L)):
            a, b = L[i], L[j]
            if a[0] == b[0] and a[1] == b[1] and len(a[2]) == len(b[2]):
                s = unify_args(a[2], b[2])
                if s is not None:
                    yield s, subst_lits(lits, s)


class Prover:
    def __init__(self, max_clauses=3000):
        self.clauses = {}          # id -> Clause
        self.seen = {}             # canonical literal set -> id
        self.usable = []           # clauses outside the set of support
        self.heap = []             # set of support, smallest clause first
        self.counter = itertools.count(1)
        self.empty = None
        self.max_clauses = max_clauses

    def add(self, lits, parents, kind, note, theta, sos):
        lits = canonical(lits)
        if is_tautology(lits) or lits in self.seen:
            return None
        cid = next(self.counter)
        c = Clause(cid, lits, tuple(parents), kind, note, theta, sos)
        self.clauses[cid] = c
        self.seen[lits] = cid
        if not lits:
            if self.empty is None:
                self.empty = c
            return c
        if sos:
            heapq.heappush(self.heap, (len(lits), cid))
        else:
            self.usable.append(c)
        for s, f in factors(lits):
            self.add(f, (cid,), "factor", "factor", fmt_theta(s), sos)
        return c

    def refute(self):
        """returns 'proved' | 'saturated' | 'limit'"""
        while self.heap and self.empty is None:
            if len(self.clauses) > self.max_clauses:
                return "limit"
            _, gid = heapq.heappop(self.heap)
            given = self.clauses[gid]
            for other in list(self.usable) + [given]:
                for pred, s, rest in resolvents(given.lits, other.lits):
                    self.add(rest, (given.id, other.id), "resolve", pred, fmt_theta(s), True)
                    if self.empty is not None:
                        return "proved"
            self.usable.append(given)
        return "proved" if self.empty is not None else "saturated"

    def proof_ids(self):
        need, stack = set(), [self.empty.id]
        while stack:
            i = stack.pop()
            if i not in need:
                need.add(i)
                stack.extend(self.clauses[i].parents)
        return sorted(need)


# ════════════════════════════════════════════════════════════════════
#  5.  OUTPUT : text proof, Graphviz, matplotlib graph
# ════════════════════════════════════════════════════════════════════
def print_proof(p):
    ids = p.proof_ids()
    num = {cid: i for i, cid in enumerate(ids, 1)}
    print("\nRESOLUTION PROOF  (negated goal  ->  empty clause)")
    print("(partner clause's variables are renamed y1, y2, ... before unifying)\n")
    for cid in ids:
        c = p.clauses[cid]
        if c.kind == "premise":
            how = c.note
        elif c.kind == "goal":
            how = "NEGATED GOAL"
        elif c.kind == "factor":
            how = f"factor of ({num[c.parents[0]]})  θ={c.theta}"
        else:
            a, b = c.parents
            how = f"resolve ({num[a]}, {num[b]}) on {c.note}  θ={c.theta}"
        print(f"({num[cid]:>2}) {fmt_clause(c.lits):<58} {how}")


def node_label(p, c, num, wrap=34):
    text = textwrap.fill(f"({num[c.id]}) {fmt_clause(c.lits, True)}", wrap, break_long_words=False)
    if c.kind == "premise":
        sub = c.note
    elif c.kind == "goal":
        sub = "negated goal"
    elif c.kind == "factor":
        sub = f"factor of ({num[c.parents[0]]})  θ={c.theta}"
    else:
        a, b = c.parents
        sub = f"R({num[a]},{num[b]}) on {c.note}\nθ={c.theta}"
    return text + "\n" + sub


def write_dot(p, path):
    ids = p.proof_ids()
    num = {cid: i for i, cid in enumerate(ids, 1)}
    lines = ["digraph resolution {", "  rankdir=TB;", '  node [shape=box, style="rounded,filled", fontname="Helvetica"];']
    colour = {"goal": "#f8c9c4", "premise": "#cfe3f8", "resolve": "#f3f3f3", "factor": "#fdf0c9"}
    for cid in ids:
        c = p.clauses[cid]
        fill = "#c7ecc7" if not c.lits else colour[c.kind]
        label = node_label(p, c, num).replace('"', "'").replace("\n", "\\n")
        lines.append(f'  n{cid} [label="{label}", fillcolor="{fill}"];')
    for cid in ids:
        for pa in set(p.clauses[cid].parents):
            lines.append(f"  n{pa} -> n{cid};")
    lines.append("}")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(lines))


def draw_graph(p, png_path, show=False):
    try:
        import matplotlib
        if not show:
            matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        from matplotlib.patches import Patch
    except ImportError:
        print("(matplotlib not installed - skipping PNG; `pip install matplotlib`)")
        return False

    ids = p.proof_ids()
    num = {cid: i for i, cid in enumerate(ids, 1)}
    C = p.clauses
    children = defaultdict(list)
    for cid in ids:
        for pa in set(C[cid].parents):
            children[pa].append(cid)

    # layers: set-of-support chain flows top -> bottom from the negated goal;
    # premises are placed just above the first clause that uses them
    layer = {}
    for cid in ids:
        if C[cid].sos:
            sp = [layer[q] for q in set(C[cid].parents) if C[q].sos]
            layer[cid] = 1 + max(sp) if sp else 0
    for cid in reversed(ids):
        if not C[cid].sos:
            layer[cid] = min(layer[ch] for ch in children[cid]) - 1
    m = min(layer.values())
    layer = {k: v - m for k, v in layer.items()}

    rows = defaultdict(list)
    for cid in ids:
        rows[layer[cid]].append(cid)
    XS, YS = 4.3, 2.3
    xs = {}
    for L in sorted(rows):
        def key(cid):
            c = C[cid]
            if c.kind == "goal":
                return (0, 0)
            px = [xs[q] for q in set(c.parents) if q in xs]
            return (1, sum(px) / len(px)) if px else (2, cid)
        rows[L].sort(key=key)
        n = len(rows[L])
        for i, cid in enumerate(rows[L]):
            xs[cid] = (i - (n - 1) / 2) * XS

    max_n = max(len(v) for v in rows.values())
    n_layers = len(rows)
    W, H = max_n * XS + 2, (n_layers - 1) * YS + 2.6
    fig, ax = plt.subplots(figsize=(max(W, 8), max(H, 4)))
    ax.set_xlim(-W / 2, W / 2)
    ax.set_ylim(-(n_layers - 1) * YS - 1.3, 1.3)
    ax.axis("off")
    ax.set_title("Resolution refutation:  negated goal  →  □ (empty clause)", fontsize=12)

    colour = {"goal": "#f8c9c4", "premise": "#cfe3f8", "resolve": "#f3f3f3", "factor": "#fdf0c9"}
    texts = {}
    for cid in ids:
        c = C[cid]
        fc = "#c7ecc7" if not c.lits else colour[c.kind]
        texts[cid] = ax.text(xs[cid], -layer[cid] * YS, node_label(p, c, num), ha="center", va="center",
                             fontsize=8, family="DejaVu Sans",
                             bbox=dict(boxstyle="round,pad=0.4", fc=fc, ec="#555555", lw=1.0))
    fig.canvas.draw()
    for cid in ids:
        for pa in set(C[cid].parents):
            ax.annotate("", xy=(xs[cid], -layer[cid] * YS), xytext=(xs[pa], -layer[pa] * YS),
                        arrowprops=dict(arrowstyle="-|>", color="#444444", lw=1.1,
                                        patchA=texts[pa].get_bbox_patch(),
                                        patchB=texts[cid].get_bbox_patch(),
                                        shrinkA=2, shrinkB=2))
    ax.legend(handles=[Patch(fc=colour["goal"], ec="#555", label="negated goal"),
                       Patch(fc=colour["premise"], ec="#555", label="premise clause"),
                       Patch(fc=colour["resolve"], ec="#555", label="resolvent"),
                       Patch(fc="#c7ecc7", ec="#555", label="empty clause □")],
              loc="lower left", fontsize=8, frameon=False)
    fig.savefig(png_path, dpi=160, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)
    return True


# ════════════════════════════════════════════════════════════════════
#  6.  DRIVER
# ════════════════════════════════════════════════════════════════════
DEMO_PREMISES = [
    "Man(Marcus)",
    "Pompeian(Marcus)",
    "forall x (Pompeian(x) -> Roman(x))",
    "Ruler(Caesar)",
    "forall x (Roman(x) -> Loyal(x, Caesar) | Hate(x, Caesar))",
    "forall x exists y Loyal(x, y)",
    "forall x forall y (Man(x) & Ruler(y) & TryAssassinate(x, y) -> ~Loyal(x, y))",
    "TryAssassinate(Marcus, Caesar)",
]
DEMO_GOAL = "Hate(Marcus, Caesar)"


def solve(premises, goal, png="resolution_graph.png", show=False, max_clauses=3000):
    conv, prover = Converter(), Prover(max_clauses)

    print("=" * 70)
    print("STEP 1 - CONVERSION TO CNF")
    print("=" * 70)
    for k, text in enumerate(premises, 1):
        parsed = Parser(text).parse()
        clauses = premise_to_clauses(parsed, conv)
        print(f"\nPremise {k}: {text}")
        print(f"   parsed as: {fmt_formula(parsed)}")
        for c in clauses:
            print("    ", fmt_clause(canonical(c)))
            prover.add(c, (), "premise", f"premise {k}", "", sos=False)

    goal_ast = Parser(goal).parse()
    print(f"\nGoal: {goal}")
    print(f"   parsed as: {fmt_formula(goal_ast)}")
    for c in negated_goal_to_clauses(goal_ast, conv):
        print("   negated:", fmt_clause(canonical(c)))
        prover.add(c, (), "goal", "negated goal", "", sos=True)

    print("\n" + "=" * 70)
    print("STEP 2 - RESOLUTION REFUTATION")
    print("=" * 70)
    status = prover.refute()

    if status == "proved":
        print_proof(prover)
        print("\n✔ Empty clause derived: the premises ENTAIL the goal.")
        dot = png.rsplit(".", 1)[0] + ".dot"
        write_dot(prover, dot)
        ok = draw_graph(prover, png, show)
        print(f"\nGraph written to: {png if ok else '(PNG skipped)'}  and  {dot}  (dot -Tpng {dot} -o proof.png)")
    elif status == "saturated":
        print("✘ No more resolvents can be generated and no empty clause was found:"
              " the goal does NOT follow from the premises.")
    else:
        print(f"? Stopped after {len(prover.clauses)} clauses without a refutation (limit reached)."
              " FOL entailment is only semi-decidable, so this is inconclusive - try --max-clauses.")
    return status, prover


def read_file(path):
    premises, goal = [], None
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if line.upper().startswith("GOAL:"):
                goal = line[5:].strip()
            else:
                premises.append(line)
    if goal is None:
        sys.exit("File must contain a line starting with  GOAL:")
    return premises, goal


def read_interactive():
    print(__doc__.split("SYNTAX")[1].split("USAGE")[0])
    print("Enter premises, one per line (blank line to finish):")
    premises = []
    while True:
        line = input(f"  premise {len(premises) + 1}> ").strip()
        if not line:
            break
        premises.append(line)
    goal = input("  goal> ").strip()
    return premises, goal


def main():
    ap = argparse.ArgumentParser(description="FOL -> CNF -> resolution refutation + proof graph")
    ap.add_argument("--demo", action="store_true", help="run the built-in example")
    ap.add_argument("--file", help="file with one premise per line and a final 'GOAL: ...' line")
    ap.add_argument("--out", default="resolution_graph.png", help="output PNG for the proof graph")
    ap.add_argument("--show", action="store_true", help="also open the graph in a window")
    ap.add_argument("--max-clauses", type=int, default=3000)
    a, unknown = ap.parse_known_args()

    if a.demo:
        premises, goal = DEMO_PREMISES, DEMO_GOAL
    elif a.file:
        premises, goal = read_file(a.file)
    else:
        premises, goal = read_interactive()
    if not goal:
        sys.exit("No goal given.")
    try:
        solve(premises, goal, a.out, a.show, a.max_clauses)
    except ParseError as e:
        sys.exit(f"Parse error: {e}")


if __name__ == "__main__":
    main()


  predicates / constants / functions : Capitalised    Man(x)  Loves(x, Mother(y))  Marcus
  variables                          : lowercase      x  y  z1
  not  ~      and  &      or  |      implies  ->      iff  <->
  quantifiers: forall x (...)   exists y (...)   forall x,y (...)   forall x forall y ...
  (a quantifier's scope extends as far right as possible, so write
   (exists y P(y)) -> Q   or   forall x ((forall y R(x,y)) -> S(x))   to limit it;
   the "parsed as:" line printed for every formula shows how it was read;
   unicode symbols  ∀ ∃ ¬ ∧ ∨ → ↔  are accepted too)
  Free variables: universally quantified in premises, existentially in the goal.


Enter premises, one per line (blank line to finish):
  premise 1> ∀x (Likes(x,Science) → Likes(x,Physics) ∨ Likes(x,Mathematics))
  premise 2> ∀x (Likes(x,Mathematics) → Likes(x,Statistics))
  premise 3> Likes(Alice,Science)
  premise 4> Likes(Bob,Physics)
  premise 5> ∀x (Likes(x,Statistics) → Likes(x,DataAnalysis))
  premise 6> ∀